# Hub Post-Processing: Line Status Aggregation

This notebook adds line status columns to the final scored hub output.

**Input files:**
1. Scored hubs CSV (output from scoring pipeline)
2. Line status CSV with columns: `LineName`, `Year`, `Status`, `Status Id`

**Output:**
- Scored hubs with additional columns: `NumLinesStatus_0`, `NumLinesStatus_1`, ..., `NumLinesStatus_7`
- Each column counts how many lines in that hub have the corresponding status

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## Step 1: Load Scored Hubs Data

Load the final output from the scoring pipeline.

In [ ]:
# Configure the path to your scored hubs file
# Option 1: Use the latest file from results directory
# scored_hubs_file = RESULTS_DIR / 'scored_hubs_20251222.csv'

# Option 2: Use a specific file
# scored_hubs_file = RESULTS_DIR / 'hubs_with_names_20251222.csv'

# Option 3: Auto-detect the latest scored file
scored_files = sorted(RESULTS_DIR.glob('scored_hubs_*.csv'))
if scored_files:
    scored_hubs_file = scored_files[-1]  # Use the latest
else:
    # Fallback to any file in results
    scored_files = sorted(RESULTS_DIR.glob('*.csv'))
    if scored_files:
        scored_hubs_file = scored_files[-1]
    else:
        raise FileNotFoundError("No scored hubs CSV file found in results directory")

print(f"Loading scored hubs from: {scored_hubs_file.name}")

# Load the scored hubs data
hubs_df = pd.read_csv(scored_hubs_file, encoding='utf-8')

print(f"\nLoaded {len(hubs_df)} hubs")
print(f"Columns: {len(hubs_df.columns)}")
print(f"\nFirst few columns: {hubs_df.columns[:10].tolist()}")

# Display sample
hubs_df.head(3)

## Step 2: Load Line Status Data

Load the CSV file containing line status information.

**Expected columns:**
- `LineName`: Name of the transit line
- `Year`: Planning year
- `Status`: Status description (e.g., "In Planning", "Under Construction", "Operational")
- `Status Id`: Numeric status identifier (0-7)

**Note:** The file should also include a column to link lines to hubs (e.g., `group` or `hub_id`)

In [ ]:
# Configure the path to your line status file
# Update this path to point to your actual file
line_status_file = DATA_DIR / 'line_status.csv'

# Alternative locations to check
if not line_status_file.exists():
    alternative_paths = [
        PROCESSED_DATA_DIR / 'line_status.csv',
        DATA_DIR / 'lines_with_status.csv',
        PROCESSED_DATA_DIR / 'lines_with_status.csv',
    ]
    
    for path in alternative_paths:
        if path.exists():
            line_status_file = path
            break

if not line_status_file.exists():
    print(f"ERROR: Line status file not found at: {line_status_file}")
    print("\nPlease provide a CSV file with the following columns:")
    print("  - group (or hub_id): Hub identifier to match with scored hubs")
    print("  - LineName: Name of the transit line")
    print("  - Year: Planning year")
    print("  - Status: Status description")
    print("  - Status Id: Numeric status identifier (0-7)")
    print("\nSave this file to one of these locations:")
    print(f"  - {DATA_DIR / 'line_status.csv'}")
    print(f"  - {PROCESSED_DATA_DIR / 'line_status.csv'}")
    raise FileNotFoundError("Line status file not found")

print(f"Loading line status from: {line_status_file.name}")

# Load the line status data
lines_df = pd.read_csv(line_status_file, encoding='utf-8')

print(f"\nLoaded {len(lines_df)} line records")
print(f"Columns: {lines_df.columns.tolist()}")

# Check for required columns
required_cols = ['LineName', 'Status Id']
missing_cols = [col for col in required_cols if col not in lines_df.columns]
if missing_cols:
    print(f"\nWARNING: Missing required columns: {missing_cols}")

# Display sample
lines_df.head()

## Step 3: Verify Status Values

Check what status values exist in the data.

In [ ]:
# Check unique status IDs
if 'Status Id' in lines_df.columns:
    unique_statuses = sorted(lines_df['Status Id'].dropna().unique())
    print(f"Unique Status IDs found: {unique_statuses}")
    print(f"Number of unique statuses: {len(unique_statuses)}")
    
    # Show distribution
    print("\nStatus distribution:")
    if 'Status' in lines_df.columns:
        status_dist = lines_df.groupby(['Status Id', 'Status']).size().reset_index(name='count')
        display(status_dist)
    else:
        print(lines_df['Status Id'].value_counts().sort_index())
else:
    print("ERROR: 'Status Id' column not found in line status data")

## Step 4: Prepare Line-to-Hub Mapping

Create a mapping between lines and hubs. This requires identifying which column links lines to hubs.

In [ ]:
# Identify the hub identifier column in both dataframes
# Common options: 'group', 'hub_id', 'group_id', 'id'

# For hubs dataframe
hub_id_col = None
for col in ['group', 'hub_id', 'group_id', 'id']:
    if col in hubs_df.columns:
        hub_id_col = col
        break

if hub_id_col is None:
    print("ERROR: Could not find hub identifier column in scored hubs data")
    print(f"Available columns: {hubs_df.columns.tolist()[:20]}")
    raise ValueError("Hub identifier column not found")

print(f"Hub identifier column in scored hubs: '{hub_id_col}'")

# For lines dataframe - check if it has a hub identifier
line_hub_col = None
for col in ['group', 'hub_id', 'group_id', 'id', hub_id_col]:
    if col in lines_df.columns:
        line_hub_col = col
        break

if line_hub_col is None:
    print("WARNING: Line status data does not have a hub identifier column")
    print("Will attempt to match lines to hubs using line names")
    print("\nChecking if hubs data has line information...")
    
    # Check if hubs have line information
    line_cols_in_hubs = [col for col in hubs_df.columns if 'line' in col.lower()]
    print(f"Line-related columns in hubs data: {line_cols_in_hubs}")
    
    if 'Line_Unique' in hubs_df.columns:
        print("\nUsing 'Line_Unique' column to match lines to hubs")
        use_line_matching = True
    else:
        raise ValueError("Cannot establish link between lines and hubs")
else:
    print(f"Hub identifier column in line status data: '{line_hub_col}'")
    use_line_matching = False

## Step 5: Create Line-to-Hub Mapping (if needed)

If the line status data doesn't have hub IDs, we need to create the mapping by matching line names.

In [ ]:
def parse_line_list(line_str):
    """
    Parse string representation of line list to actual list.
    Handles formats like: "['Line1', 'Line2']" or "Line1, Line2"
    """
    if pd.isna(line_str) or line_str == '':
        return []
    
    line_str = str(line_str)
    
    # Remove brackets and quotes
    line_str = line_str.replace("'", "").replace('[', '').replace(']', '').replace('"', '')
    
    # Split by comma
    lines = [x.strip() for x in line_str.split(',') if x.strip()]
    
    return lines

if use_line_matching:
    print("Creating line-to-hub mapping...")
    
    # Parse line lists in hubs dataframe
    if 'Line_Unique' in hubs_df.columns:
        # Check if it's already a list or needs parsing
        if isinstance(hubs_df['Line_Unique'].iloc[0], str):
            hubs_df['Line_Unique_Parsed'] = hubs_df['Line_Unique'].apply(parse_line_list)
        else:
            hubs_df['Line_Unique_Parsed'] = hubs_df['Line_Unique']
    
    # Create a mapping: LineName -> list of hub IDs
    line_to_hub_map = {}
    
    for idx, row in hubs_df.iterrows():
        hub_id = row[hub_id_col]
        lines = row.get('Line_Unique_Parsed', [])
        
        if not isinstance(lines, list):
            lines = [lines] if pd.notna(lines) else []
        
        for line in lines:
            if pd.notna(line) and line:
                if line not in line_to_hub_map:
                    line_to_hub_map[line] = []
                line_to_hub_map[line].append(hub_id)
    
    print(f"Created mapping for {len(line_to_hub_map)} unique lines")
    
    # Add hub IDs to lines dataframe
    def get_hub_ids(line_name):
        """Get hub IDs for a given line name."""
        if pd.isna(line_name):
            return []
        return line_to_hub_map.get(str(line_name).strip(), [])
    
    lines_df['hub_ids'] = lines_df['LineName'].apply(get_hub_ids)
    
    # Explode so each line-hub pair gets its own row
    lines_expanded = lines_df.explode('hub_ids').copy()
    lines_expanded = lines_expanded[lines_expanded['hub_ids'].notna()].copy()
    lines_expanded.rename(columns={'hub_ids': hub_id_col}, inplace=True)
    
    print(f"Expanded to {len(lines_expanded)} line-hub pairs")
    
    # Use this for aggregation
    lines_for_aggregation = lines_expanded
else:
    print("Using existing hub identifier in line status data")
    lines_for_aggregation = lines_df.copy()
    
    # Ensure column names match
    if line_hub_col != hub_id_col:
        lines_for_aggregation.rename(columns={line_hub_col: hub_id_col}, inplace=True)

print(f"\nReady for aggregation: {len(lines_for_aggregation)} records")
lines_for_aggregation.head()

## Step 6: Aggregate Line Counts by Status

For each hub, count how many lines have each status (0-7).

In [ ]:
def aggregate_line_status_counts(lines_df, hub_id_col, status_col='Status Id'):
    """
    Aggregate line counts by status for each hub.
    
    Args:
        lines_df: DataFrame with line status data
        hub_id_col: Column name for hub identifier
        status_col: Column name for status ID
    
    Returns:
        DataFrame with columns: hub_id, NumLinesStatus_0, NumLinesStatus_1, ...
    """
    # Group by hub and status, count lines
    status_counts = lines_df.groupby([hub_id_col, status_col]).size().reset_index(name='count')
    
    # Pivot to wide format
    status_wide = status_counts.pivot(index=hub_id_col, columns=status_col, values='count')
    
    # Rename columns to NumLinesStatus_X
    status_wide.columns = [f'NumLinesStatus_{int(col)}' for col in status_wide.columns]
    
    # Reset index
    status_wide = status_wide.reset_index()
    
    # Fill NaN with 0 (hubs with no lines in that status)
    status_cols = [col for col in status_wide.columns if col.startswith('NumLinesStatus_')]
    status_wide[status_cols] = status_wide[status_cols].fillna(0).astype(int)
    
    return status_wide

# Aggregate line status counts
print("Aggregating line counts by status...")
status_aggregated = aggregate_line_status_counts(
    lines_for_aggregation, 
    hub_id_col=hub_id_col,
    status_col='Status Id'
)

print(f"\nAggregated status counts for {len(status_aggregated)} hubs")
print(f"Status columns created: {[col for col in status_aggregated.columns if col.startswith('NumLinesStatus_')]}")

# Display sample
status_aggregated.head(10)

## Step 7: Ensure All Status Columns (0-7) Exist

Add any missing status columns and set them to 0.

In [ ]:
# Ensure all status columns 0-7 exist
expected_status_cols = [f'NumLinesStatus_{i}' for i in range(8)]

for col in expected_status_cols:
    if col not in status_aggregated.columns:
        print(f"Adding missing column: {col}")
        status_aggregated[col] = 0

# Reorder columns to have status columns in order
other_cols = [col for col in status_aggregated.columns if not col.startswith('NumLinesStatus_')]
status_cols = sorted([col for col in status_aggregated.columns if col.startswith('NumLinesStatus_')])

status_aggregated = status_aggregated[other_cols + status_cols]

print(f"\nFinal status columns: {status_cols}")
status_aggregated.head()

## Step 8: Merge with Scored Hubs Data

Add the status columns to the original scored hubs dataframe.

In [ ]:
# Merge status counts with original hubs data
print(f"Merging status counts with scored hubs data...")
print(f"Hubs before merge: {len(hubs_df)}")

hubs_with_status = hubs_df.merge(
    status_aggregated,
    on=hub_id_col,
    how='left'
)

print(f"Hubs after merge: {len(hubs_with_status)}")

# Fill NaN with 0 for hubs that don't have any lines in status data
status_cols = [col for col in hubs_with_status.columns if col.startswith('NumLinesStatus_')]
hubs_with_status[status_cols] = hubs_with_status[status_cols].fillna(0).astype(int)

print(f"\nStatus columns added: {len(status_cols)}")
print(f"Total columns now: {len(hubs_with_status.columns)}")

# Display sample with new columns
display_cols = [hub_id_col, 'HubType', 'TotalDemand'] + status_cols
available_display_cols = [col for col in display_cols if col in hubs_with_status.columns]
hubs_with_status[available_display_cols].head(10)

## Step 9: Add Total Line Count Column

Add a column with the total count of all lines across all statuses.

In [ ]:
# Calculate total lines across all statuses
hubs_with_status['TotalLinesAllStatuses'] = hubs_with_status[status_cols].sum(axis=1)

print(f"Total lines column added")
print(f"\nSummary statistics for total lines:")
print(hubs_with_status['TotalLinesAllStatuses'].describe())

# Show hubs with most lines
print(f"\nTop 10 hubs by total line count:")
top_by_lines = hubs_with_status.nlargest(10, 'TotalLinesAllStatuses')
display_cols = [hub_id_col, 'TotalLinesAllStatuses'] + status_cols
if 'HubName' in hubs_with_status.columns:
    display_cols.insert(1, 'HubName')
if 'address' in hubs_with_status.columns:
    display_cols.insert(1, 'address')

available_display_cols = [col for col in display_cols if col in hubs_with_status.columns]
top_by_lines[available_display_cols]

## Step 10: Summary Statistics by Status

Display distribution of lines by status across all hubs.

In [ ]:
# Calculate summary statistics for each status
print("Summary Statistics by Status")
print("=" * 80)

summary_data = []
for col in status_cols:
    status_num = col.replace('NumLinesStatus_', '')
    
    summary = {
        'Status ID': status_num,
        'Total Lines': hubs_with_status[col].sum(),
        'Hubs with Lines': (hubs_with_status[col] > 0).sum(),
        'Max Lines (one hub)': hubs_with_status[col].max(),
        'Avg Lines (all hubs)': hubs_with_status[col].mean(),
        'Avg Lines (non-zero hubs)': hubs_with_status[hubs_with_status[col] > 0][col].mean() if (hubs_with_status[col] > 0).any() else 0
    }
    summary_data.append(summary)

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Overall totals
print(f"\nTotal lines across all hubs and statuses: {hubs_with_status['TotalLinesAllStatuses'].sum()}")
print(f"Average lines per hub: {hubs_with_status['TotalLinesAllStatuses'].mean():.2f}")
print(f"Hubs with at least one line: {(hubs_with_status['TotalLinesAllStatuses'] > 0).sum()}")

## Step 11: Export Results

Save the enriched data with line status columns to a new CSV file.

In [ ]:
# Create output filename with timestamp
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
output_file = RESULTS_DIR / f"hubs_with_line_status_{timestamp}.csv"

# Reorder columns to put new status columns near the beginning
# Priority: hub ID, name/address, scores, status columns, then everything else
priority_cols = [hub_id_col]

if 'HubName' in hubs_with_status.columns:
    priority_cols.append('HubName')
if 'HubType' in hubs_with_status.columns:
    priority_cols.append('HubType')
if 'Overall_Rank' in hubs_with_status.columns:
    priority_cols.append('Overall_Rank')
if 'Average_Simulated_Score' in hubs_with_status.columns:
    priority_cols.append('Average_Simulated_Score')
if 'TotalDemand' in hubs_with_status.columns:
    priority_cols.append('TotalDemand')

# Add total lines and status columns
priority_cols.append('TotalLinesAllStatuses')
priority_cols.extend(status_cols)

# Add remaining columns
other_cols = [col for col in hubs_with_status.columns if col not in priority_cols]
final_col_order = [col for col in priority_cols if col in hubs_with_status.columns] + other_cols

# Reorder and export
hubs_export = hubs_with_status[final_col_order].copy()

# Sort by rank if available
if 'Overall_Rank' in hubs_export.columns:
    hubs_export = hubs_export.sort_values('Overall_Rank')

hubs_export.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Results exported to: {output_file}")
print(f"\nTotal hubs: {len(hubs_export)}")
print(f"Total columns: {len(hubs_export.columns)}")
print(f"\nNew columns added:")
print(f"  - TotalLinesAllStatuses")
for col in status_cols:
    print(f"  - {col}")

## Summary

Post-processing complete!

### Outputs:
1. **New columns added** (8 status columns + 1 total):
   - `NumLinesStatus_0` through `NumLinesStatus_7`: Count of lines per status
   - `TotalLinesAllStatuses`: Total line count across all statuses

2. **CSV file exported** with all original columns plus new line status columns

3. **Summary statistics** showing distribution of lines by status

### Next Steps:
- Review the output file to verify results
- Use the status columns for further analysis or visualization
- Consider adding visualizations (e.g., stacked bar charts by hub type and status)